<a href="https://colab.research.google.com/github/raphy0316/Insight-Hub-Models/blob/imagebind/LLImageBindTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===============================================
# STEP 1. 환경 설정 & 패키지 설치 (GPU + 로컬 전용)
#  - ImageBind: 모델/전처리
#  - timm, librosa, soundfile: 오디오/모델 의존성
#  - ffmpeg: 오디오 디코딩
#  - gradio: 간단한 웹 UI
# ===============================================
!nvidia-smi  # GPU 확인 (No CUDA면 런타임 유형을 GPU로)

%pip -q install "git+https://github.com/facebookresearch/ImageBind.git" timm==0.9.2 soundfile librosa ffmpeg-python gradio
!apt -yqq install ffmpeg


Thu Nov  6 22:26:07 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             33W /   70W |    9768MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# ===============================================
# STEP 2. Colab 로컬 경로만 사용 (영구 보관 X)
#  - BASE: 프로젝트 루트
#  - ART : 임베딩(artifacts) 저장 폴더
#  - DATA: 오디오(mp3 등) 저장 폴더
#  - EXTD: (선택) 데이터셋 압축/해제 보관 폴더
# ===============================================
import os

BASE  = "/content/ImageBind"
ART   = f"{BASE}/artifacts"
DATA  = f"{BASE}/data/audio_src"
EXTD  = f"{BASE}/external_datasets"

for d in (BASE, ART, DATA, EXTD):
    os.makedirs(d, exist_ok=True)

print("BASE:", BASE)
print("Artifacts:", ART)
print("Audio src:", DATA)


BASE: /content/ImageBind
Artifacts: /content/ImageBind/artifacts
Audio src: /content/ImageBind/data/audio_src


In [ ]:
# ===============================================
# STEP 3. FMA small 일부 다운로드 (테스트용)
#  - 런타임 종료 시 전부 삭제.
#  - 이미 mp3가 있으면 이 셀은 건너뛰기.
# ===============================================
import os, glob, random, urllib.request, zipfile, shutil

url = "https://os.unil.cloud.switch.ch/fma/fma_small.zip"
zip_path = f"{EXTD}/fma_small.zip"
ext_dir  = f"{EXTD}/fma_small_extracted"
max_mp3  = 500  # 필요에 맞게 조절(300~1000 추천)

print("Existing mp3 before:", len(glob.glob(f"{DATA}/*.mp3")))

if len(glob.glob(f"{DATA}/*.mp3")) == 0:
    if not os.path.exists(zip_path):
        print("⬇️ Downloading fma_small.zip ...")
        urllib.request.urlretrieve(url, zip_path)

    if not os.path.exists(ext_dir):
        print("📦 Extracting ...")
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(ext_dir)

    SRC_ROOT = f"{ext_dir}/fma_small"
    files = glob.glob(f"{SRC_ROOT}/**/*.mp3", recursive=True)
    sample = random.sample(files, min(max_mp3, len(files)))
    for src in sample:
        shutil.copy2(src, os.path.join(DATA, os.path.basename(src)))

print("✅ Now in data/audio_src:", len(glob.glob(f'{DATA}/*.mp3')))


Existing mp3 before: 0
⬇️ Downloading fma_small.zip ...
📦 Extracting ...
✅ Now in data/audio_src: 500


In [ ]:
# ===============================================
# STEP 4. 오디오 임베딩 인덱스 생성 (GPU + float16 저장)
# ===============================================
from pathlib import Path
import numpy as np
import torch
from imagebind.models import imagebind_model
from imagebind.models.imagebind_model import ModalityType
from imagebind import data as ib_data

def get_device():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"🚀 Using device: {device}")
    return device

def l2_normalize(x: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    return x / (x.norm(dim=-1, keepdim=True) + eps)

def build_audio_index(audio_dir: str, out_dir: str, batch_size: int = 8):
    device = get_device()
    model = imagebind_model.imagebind_huge(pretrained=True)
    model.eval().to(device)

    audio_paths = sorted([str(p) for p in Path(audio_dir).rglob("*.mp3")])
    if not audio_paths:
        raise RuntimeError(f"No mp3 files found in {audio_dir}")

    embs = []
    with torch.no_grad():
        for i in range(0, len(audio_paths), batch_size):
            chunk = audio_paths[i:i+batch_size]
            batch = {ModalityType.AUDIO: ib_data.load_and_transform_audio_data(chunk, device)}
            feats = model(batch)[ModalityType.AUDIO]      # (b, D) on GPU
            feats = l2_normalize(feats).cpu().numpy().astype(np.float16)
            embs.append(feats)
            print(f"[{i+len(chunk):>5}/{len(audio_paths)}] encoded")

    embs = np.concatenate(embs, axis=0)
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    np.save(f"{out_dir}/emb_audio_float16.npy", embs)
    np.save(f"{out_dir}/audio_paths.npy", np.array(audio_paths, dtype=object))

    print("\n✅ Index built!")
    print("Saved embeddings:", f"{out_dir}/emb_audio_float16.npy", embs.shape, embs.dtype)
    print("Saved paths:", f"{out_dir}/audio_paths.npy", len(audio_paths))

# 실행 (필요하면 batch_size 조절)
build_audio_index(DATA, ART, batch_size=8)


🚀 Using device: cuda


/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

[    8/500] encoded
[   16/500] encoded
[   24/500] encoded
[   32/500] encoded
[   40/500] encoded
[   48/500] encoded
[   56/500] encoded
[   64/500] encoded
[   72/500] encoded
[   80/500] encoded
[   88/500] encoded
[   96/500] encoded
[  104/500] encoded
[  112/500] encoded
[  120/500] encoded
[  128/500] encoded
[  136/500] encoded
[  144/500] encoded
[  152/500] encoded
[  160/500] encoded
[  168/500] encoded
[  176/500] encoded
[  184/500] encoded
[  192/500] encoded
[  200/500] encoded
[  208/500] encoded
[  216/500] encoded
[  224/500] encoded
[  232/500] encoded
[  240/500] encoded
[  248/500] encoded
[  256/500] encoded
[  264/500] encoded
[  272/500] encoded
[  280/500] encoded
[  288/500] encoded
[  296/500] encoded
[  304/500] encoded
[  312/500] encoded
[  320/500] encoded
[  328/500] encoded
[  336/500] encoded
[  344/500] encoded
[  352/500] encoded
[  360/500] encoded
[  368/500] encoded
[  376/500] encoded
[  384/500] encoded
[  392/500] encoded
[  400/500] encoded


In [ ]:
# ===============================================
# STEP 4. 모델/임베딩 1회 로드 (속도 최적화)
# ===============================================
import torch
import tempfile, imageio
import numpy as np
from imagebind.models import imagebind_model
from imagebind.models.imagebind_model import ModalityType
from imagebind import data as ib_data

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL = imagebind_model.imagebind_huge(pretrained=True).eval().to(DEVICE)
EMB_AUDIO = np.load(f"{ART}/emb_audio_float16.npy").astype(np.float32)
AUDIO_PATHS = np.load(f"{ART}/audio_paths.npy", allow_pickle=True).tolist()

# GPU 워밍업 (CUDA 커널 초기화 → 첫 추론 속도 향상)
with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
    imageio.imwrite(tmp.name, np.zeros((64, 64, 3), dtype=np.uint8))
    with torch.no_grad():
        _batch = {ModalityType.VISION: ib_data.load_and_transform_vision_data([tmp.name], DEVICE)}
        _ = MODEL(_batch)[ModalityType.VISION]

print("✅ Model warm-up done.")


✅ Model warm-up done.


In [ ]:
# ===============================================
# STEP 5. 추천 함수 (Top-1 문자열 경로 반환)
# ===============================================
import os
import numpy as np
import pandas as pd
import torch
from imagebind.models.imagebind_model import ModalityType
from imagebind import data as ib_data

def ui_recommend(image, k):
    import tempfile, imageio
    if image is None:
        return None, None

    # Gradio가 ndarray로 전달하므로 임시 파일로 저장
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
        imageio.imwrite(tmp.name, image)
        img_path = tmp.name

    # 이미지 → 임베딩 변환
    with torch.no_grad():
        batch = {ModalityType.VISION: ib_data.load_and_transform_vision_data([img_path], DEVICE)}
        feats = MODEL(batch)[ModalityType.VISION]
        feats = feats / (feats.norm(dim=-1, keepdim=True) + 1e-8)
        q = feats.cpu().numpy()[0].astype(np.float32)

    # 유사도 계산
    qn = q / (np.linalg.norm(q) + 1e-8)
    En = EMB_AUDIO / (np.linalg.norm(EMB_AUDIO, axis=1, keepdims=True) + 1e-8)
    sim = (En @ qn).astype(np.float32)

    # Top-K 결과
    k = int(k)
    idx = np.argsort(-sim)[:k]
    df = pd.DataFrame({
        "rank": np.arange(1, k+1),
        "audio_path": [str(AUDIO_PATHS[i]) for i in idx],   # 문자열로 변환
        "similarity": [float(sim[i]) for i in idx]
    })

    # Top-1 경로 반환
    top1 = str(AUDIO_PATHS[idx[0]]) if k > 0 else None
    if top1 and not os.path.exists(top1):
        print(f"⚠️ File not found: {top1}")
        top1 = None

    return df, top1


In [ ]:
# ===============================================
# STEP 6. Gradio UI (파일 경로 방식: 재생 복구)
# ===============================================
import gradio as gr

def ui_handler(image, k):
    return ui_recommend(image, k)   # (DataFrame, top1_path)

with gr.Blocks(title="Image→Music Recommender (Fast)") as demo:
    gr.Markdown("## 🎨 Image → 🎵 Music (Top-K)\nUpload an image and get matching songs.")
    with gr.Row():
        with gr.Column(scale=1):
            img = gr.Image(type="numpy", label="Upload Image")
            k = gr.Slider(1, 10, value=5, step=1, label="Top-K")
            run = gr.Button("Recommend")
        with gr.Column(scale=2):
            out_table = gr.Dataframe(
                headers=["rank","audio_path","similarity"],
                label="Recommendations"
            )
            # ✅ 파일 경로 방식으로 오디오 재생 복구
            out_audio = gr.Audio(type="filepath", label="Top-1 Preview", interactive=False)

    # (df, top1_path) 두 개의 출력을 연결
    run.click(fn=ui_handler, inputs=[img, k], outputs=[out_table, out_audio])

# Colab에서는 share=True가 안정적 (localhost 에러 방지)
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://23525c83ceaf8e0de4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
